# IRWA Final Project — Part 3: Ranking & Filtering

**Group:** Xavier Gutiérrez (u233714), Pau Martínez (u232318), Mikel Areta (u232868)

We reuse the pre-processing (Part 1), the inverted index and the conjunctive (AND) search (Part 2), and compare four ways of ranking the documents that contain **all** the query terms:

1. TF-IDF + cosine similarity (Part 2)
2. BM25
3. Our own score (text relevance + title match + rating + discount + stock)
4. Word2vec + cosine similarity

In Part 2 (section 3c) we listed the main problems of our TF-IDF engine: all fields weigh the same, many ties (78.8% of titles are repeated), numeric fields not used, no notion of synonyms. BM25, our score and word2vec address these one by one.

In [ ]:
import json, math, re, string
from collections import Counter, defaultdict

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

for res in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(res, quiet=True)

### Pre-processing (same as Parts 1 and 2)
Documents and queries are cleaned with the same `preprocess` function. Each product is searched through one text: title + description + useful product_details + brand + sub_category.
We also keep the title tokens separately and convert the numeric fields with `to_number` (Part 1, section 1.5), because our own score uses them.

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
PUNCT_TABLE = str.maketrans({c: " " for c in string.punctuation})


def preprocess(text):
    if not isinstance(text, str):
        return []
    text = text.encode("ascii", "ignore").decode("ascii").lower()
    text = re.sub(r"'s\b", "", text)                        # women's -> women
    text = re.sub(r"\bt[\s-]?shirts?\b", "tshirt", text)    # t-shirt / t shirt -> tshirt
    tokens = word_tokenize(text.translate(PUNCT_TABLE))
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1 and not any(c.isdigit() for c in t)]
    return [STEMMER.stem(t) for t in tokens]


with open("../../data/fashion_products_dataset.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))

DROP_DETAIL_KEYS = {"Style Code", "Fabric Care", "Sales Package", "Country of Origin", "Pack of", "Reversible",
                    "Size", "Other Details", "Model Name", "Suitable For", "Brand Color", "Brand Fit", "Sleeve Type"}
CATEGORIES = [c for c in df["category"].unique() if " " in c]


def details_to_text(details):
    return " ".join(str(v) for d in details for k, v in d.items() if k not in DROP_DETAIL_KEYS)


def normalize_sub_category(value):   # 'Roy Clothing and Accessories' -> 'Clothing and Accessories'
    return next((c for c in CATEGORIES if value.endswith(c) and value != c), value)


def to_number(value):
    # '2,999' -> 2999.0, '69% off' -> 69.0, '' -> NaN (from Part 1)
    m = re.search(r"\d+(\.\d+)?", str(value).replace(",", ""))
    return float(m.group()) if m else np.nan


df["sub_category"] = df["sub_category"].map(normalize_sub_category)
df["title_tokens"] = df["title"].map(preprocess)
df["tokens"] = (df["title_tokens"] + df["description"].map(preprocess)
                + df["product_details"].map(lambda d: preprocess(details_to_text(d)))
                + df["brand"].map(preprocess) + df["sub_category"].map(preprocess))
for col in ["selling_price", "actual_price", "discount", "average_rating"]:
    df[col] = df[col].map(to_number)
df["out_of_stock"] = df["out_of_stock"].astype(bool)

docs = df.set_index("pid")
titles = docs["title"].to_dict()
print("Documents:", len(df))

### Inverted index + conjunctive (AND) search (from Part 2)
A document is returned only if it contains **all** the query terms (intersection of the posting lists).

In [ ]:
index = defaultdict(list)   # term -> list of pids
tf = {}                     # pid  -> {term: count}
for pid, tokens in zip(df["pid"], df["tokens"]):
    tf[pid] = Counter(tokens)
    for term in tf[pid]:
        index[term].append(pid)
N = len(tf)


def conjunctive_search(query):
    terms = preprocess(query)
    if not terms:
        return set()
    return set.intersection(*[set(index.get(t, [])) for t in terms])


test_queries = {
    3: "women hooded jacket",
    4: "men running shoes",
    5: "women yellow kurta",
    6: "men regular fit formal shirt",
    7: "men sports shorts",
}
print("Vocabulary size:", len(index))
for qid, q in test_queries.items():
    print(f"{qid}. {q:30} -> {len(conjunctive_search(q))} results")

---
# 1. Scores

## 1a. TF-IDF + cosine similarity (from Part 2)

$$w_{t,d} = (1 + \log tf_{t,d}) \cdot \log\frac{N}{df_t} \qquad score(q,d) = \frac{\sum_t w_{t,q}\, w_{t,d}}{\lVert q \rVert\, \lVert d \rVert}$$

In [ ]:
idf = {term: math.log(N / len(pids)) for term, pids in index.items()}


def weight(count, term):
    return (1 + math.log(count)) * idf[term]


# length of every document vector (computed once)
doc_norm = {pid: math.sqrt(sum(weight(c, t) ** 2 for t, c in counts.items())) or 1.0 for pid, counts in tf.items()}


def tfidf_score(query_terms, pid):
    q = Counter(t for t in query_terms if t in idf)
    q_norm = math.sqrt(sum(weight(c, t) ** 2 for t, c in q.items())) or 1.0
    dot = sum(weight(c, t) * weight(tf[pid][t], t) for t, c in q.items() if t in tf[pid])
    return dot / (q_norm * doc_norm[pid])

## 1b. BM25

$$BM25(q,d) = \sum_{t \in q} IDF(t) \cdot \frac{tf_{t,d}\,(k_1+1)}{tf_{t,d} + k_1\left(1 - b + b\,\frac{|d|}{avgdl}\right)} \qquad IDF(t) = \log\left(1 + \frac{N - df_t + 0.5}{df_t + 0.5}\right)$$

- $k_1 = 1.2$ controls **term-frequency saturation**: the 2nd, 3rd… repetition of a word adds less and less.
- $b = 0.75$ controls **length normalisation**: documents longer than the average are penalised, but only partially.
- We use the standard values; $|d|$ is the number of tokens of the product and $avgdl$ the average over the collection.

In [ ]:
K1, B = 1.2, 0.75
doc_len = {pid: sum(counts.values()) for pid, counts in tf.items()}
avg_len = sum(doc_len.values()) / N
bm25_idf = {term: math.log(1 + (N - len(pids) + 0.5) / (len(pids) + 0.5)) for term, pids in index.items()}


def bm25_score(query_terms, pid):
    score = 0.0
    for t in set(query_terms):
        f = tf[pid].get(t, 0)
        if f:
            norm = K1 * (1 - B + B * doc_len[pid] / avg_len)
            score += bm25_idf[t] * f * (K1 + 1) / (f + norm)
    return score


print("Average document length:", round(avg_len, 1), "tokens")

## 1c. Our own score

$$ours(q,d) = \big(0.6\cdot \widehat{BM25}(q,d) + 0.2\cdot title\_match(q,d) + 0.15\cdot rating(d) + 0.05\cdot discount(d)\big) \times stock(d)$$

| Component | Definition | Why |
|---|---|---|
| $\widehat{BM25}$ | BM25 divided by the best BM25 among the results of the query (0–1) | Text relevance stays the main signal (60%) |
| title_match | fraction of the query terms that appear in the **title** | In Part 1 (section 5) relevant documents matched 80% of the query terms in the title vs 46% for non-relevant ones: the title is the most discriminative field |
| rating | (average_rating − 1) / 4; a missing rating gets the corpus mean | Better-rated products are more likely to satisfy the user; a product without reviews is not a badly rated product (Part 1, section 4) |
| discount | discount / 100; missing → 0 | Small boost for good deals. Low weight: a discount says nothing about relevance |
| stock | ×0.5 if out of stock, ×1 otherwise | The user cannot buy it, so we push it down without hiding it |

All components are in [0, 1], so the weights are directly comparable. Since the AND search already guarantees that every result contains all the query terms, the non-text signals only re-order documents that are all textually valid — and they break the many ties caused by repeated titles.

In [ ]:
mean_rating = df["average_rating"].mean()


def our_score(query_terms, pid, bm25_max):
    # 0.6 text (BM25, normalised per query) + 0.2 title match + 0.15 rating + 0.05 discount, halved if out of stock
    d = docs.loc[pid]
    q = set(query_terms)
    text = bm25_score(query_terms, pid) / bm25_max if bm25_max else 0.0
    title_match = len(q & set(d["title_tokens"])) / len(q)
    rating = d["average_rating"] if not np.isnan(d["average_rating"]) else mean_rating
    rating = (rating - 1) / 4                                   # 1..5 -> 0..1
    discount = 0.0 if np.isnan(d["discount"]) else d["discount"] / 100
    score = 0.6 * text + 0.2 * title_match + 0.15 * rating + 0.05 * discount
    return score * (0.5 if d["out_of_stock"] else 1.0)


print("Mean rating used for products without rating:", round(mean_rating, 2))

### One search function for every method
AND search first, then the matching documents are sorted with the chosen score. Ties are broken by pid, so the ranking is always the same.

In [ ]:
def rank(query, method="tfidf", top_k=20):
    # AND search, then rank with 'tfidf', 'bm25', 'ours' or 'w2v'
    terms = preprocess(query)
    pids = conjunctive_search(query)
    if method == "tfidf":
        scores = {p: tfidf_score(terms, p) for p in pids}
    elif method == "bm25":
        scores = {p: bm25_score(terms, p) for p in pids}
    elif method == "ours":
        bm25_max = max((bm25_score(terms, p) for p in pids), default=0.0)
        scores = {p: our_score(terms, p, bm25_max) for p in pids}
    elif method == "w2v":
        scores = {p: w2v_score(terms, p) for p in pids}
    else:
        raise ValueError(method)
    results = sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:top_k]
    return pd.DataFrame({"rank": range(1, len(results) + 1),
                         "pid": [p for p, _ in results],
                         "score": [round(s, 4) for _, s in results],
                         "title": [titles[p] for p, _ in results],
                         "length": [doc_len[p] for p, _ in results],
                         "rating": [docs.at[p, "average_rating"] for p, _ in results],
                         "discount": [docs.at[p, "discount"] for p, _ in results],
                         "out_of_stock": [docs.at[p, "out_of_stock"] for p, _ in results]})


METHODS = ["tfidf", "bm25", "ours"]
for q in test_queries.values():
    for m in METHODS:
        print(f"\nQuery: {q} | method: {m}")
        print(rank(q, m, top_k=10).to_string(index=False))

### Comparing the rankings
**Overlap@10** = share of the top-10 documents that two methods have in common (1 = same set, maybe in a different order).

In [ ]:
def overlap_at_k(a, b, k=10):
    return len(set(a[:k]) & set(b[:k])) / k


rows = []
for q in test_queries.values():
    top = {m: rank(q, m, top_k=10)["pid"].tolist() for m in METHODS}
    rows.append({"query": q,
                 "TF-IDF vs BM25": overlap_at_k(top["tfidf"], top["bm25"]),
                 "TF-IDF vs ours": overlap_at_k(top["tfidf"], top["ours"]),
                 "BM25 vs ours": overlap_at_k(top["bm25"], top["ours"])})
pd.DataFrame(rows).set_index("query")

Side-by-side titles of the top 10, to see the differences quickly:

In [ ]:
def side_by_side(query, k=10):
    return pd.DataFrame({m: rank(query, m, k)["title"].str[:45].tolist() for m in METHODS},
                        index=pd.RangeIndex(1, k + 1, name="rank"))


for q in test_queries.values():
    print(f"\nQuery: {q}")
    print(side_by_side(q).to_string())

Which documents move the most between TF-IDF and BM25, and how long they are (number of tokens; the average is printed above). This shows the effect of the different length normalisation.

In [ ]:
def rank_changes(query, k=10):
    full = {m: rank(query, m, top_k=None) for m in ["tfidf", "bm25"]}
    pos = {m: dict(zip(r["pid"], r["rank"])) for m, r in full.items()}
    top = set(full["tfidf"]["pid"][:k]) | set(full["bm25"]["pid"][:k])
    out = pd.DataFrame([{"pid": p, "title": titles[p][:45], "length": doc_len[p],
                         "tfidf_rank": pos["tfidf"][p], "bm25_rank": pos["bm25"][p]} for p in top])
    out["moved"] = out["tfidf_rank"] - out["bm25_rank"]          # > 0: BM25 ranks it higher
    return out.reindex(out["moved"].abs().sort_values(ascending=False).index).head(8)


for q in test_queries.values():
    print(f"\nQuery: {q}")
    print(rank_changes(q).to_string(index=False))

### TF-IDF vs BM25: how the ranking differs

*(The concrete examples below refer to the outputs above; re-check them if you change the code.)*

**Why the rankings differ**
- **Length normalisation.** The cosine divides by the norm of the whole document vector, so short documents (only a title and a few details — 39.7% of the products have no description, Part 1) get high scores even if the query terms are not especially important in them. BM25 normalises only partially ($b = 0.75$) and relative to the average length, so a longer product that mentions the query terms several times is not penalised as much. In the `rank_changes` tables, the documents that BM25 moves up are mostly the longer ones (compare the `length` column with the average length).
- **Term-frequency saturation.** In TF-IDF every repetition still adds weight ($1+\log tf$ keeps growing). In BM25 the contribution of a term saturates quickly ($k_1 = 1.2$): repeating *shirt* six times in a description does not beat a product that has it once in the title and once in the details.
- **IDF.** Both give more weight to rare terms (*kurta*, *hooded*) than to frequent ones (*men*, *women*). BM25's IDF is smoothed, so very common words contribute almost nothing.
- **Example — men running shoes.** In Part 2, TF-IDF put a **women's** shoe first (its sub_category is "Men's Footwear", and it is a very short document, so the cosine gives it a high score). Check the first table above to see where BM25 and our score place it.
- **Overlap.** The overlap table shows how much the top 10 changes per query: queries made of very common words (e.g. *men regular fit formal shirt*) change the most, because all the results contain the same words and only tf and length decide the order.

| | TF-IDF + cosine | BM25 |
|---|---|---|
| **Pros** | Simple and well understood; scores between 0 and 1 (comparable between queries); the same vectors can be used for document–document similarity | Usually more accurate in practice; tf saturation avoids rewarding keyword repetition; tunable length normalisation ($k_1$, $b$); standard baseline (Lucene / Elasticsearch) |
| **Cons** | Over-rewards very short documents and repeated words; no parameters to adapt to the collection | Scores are not bounded (cannot be compared across queries); $k_1$ and $b$ should be tuned per collection; still a bag of words (no synonyms, no word order) |

### Our score: justification, pros and cons

**Justification.** It follows the plan we wrote in Parts 1 and 2: numeric fields used as ranking signals, missing ratings not treated as 0, a higher weight for the title, and the many ties broken with rating, discount and stock. BM25 (not TF-IDF) is the text part because of the advantages above.

- **Pros:** uses information that the text cannot give (quality, price, availability); the title match fixes cases where a query word appears only in the description (e.g. *hooded* in a sweatshirt description); breaks ties between products with identical titles; text relevance still dominates (60%), and AND search guarantees all terms are present, so a 5-star but unrelated product cannot reach the top; easy to explain to a user.
- **Cons:** weights chosen by hand, not learned (with more labelled data they could be learned with learning-to-rank); popularity bias — well-rated products get more visibility and therefore more sales and ratings; the rating ignores the **number of reviews**, which the dataset does not have (a 5.0 from one review counts like a 5.0 from a thousand); discounts can be inflated by sellers raising the actual price; a fixed ×0.5 stock penalty may be too strong or too weak depending on the user.

---
# 2. Word2vec + cosine similarity

We train a skip-gram word2vec model on the pre-processed tokens of all products (the same stemmed terms used for the queries). A text is represented by the **average of the vectors of its words**:

$$\vec{v}(text) = \frac{1}{n}\sum_{i=1}^{n} \vec{v}(w_i)$$

Words not in the vocabulary (seen fewer than 2 times) are skipped. The documents returned by the AND search are ranked by the cosine similarity between the query vector and the document vector.

In [ ]:
sentences = df["tokens"].tolist()
w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=1, epochs=10, workers=1, seed=42)
print("Word2vec vocabulary:", len(w2v.wv.key_to_index), "words")


def text_vector(tokens):
    # average of the word vectors of the tokens that are in the vocabulary
    vectors = [w2v.wv[t] for t in tokens if t in w2v.wv.key_to_index]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)


doc_vectors = {pid: text_vector(tokens) for pid, tokens in zip(df["pid"], df["tokens"])}


def cosine(a, b):
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / norm) if norm else 0.0


def w2v_score(query_terms, pid):
    return cosine(text_vector(query_terms), doc_vectors[pid])

Sanity check: the nearest words of a few terms should be related clothing words.

In [ ]:
for word in ["jacket", "kurta", "shoe"]:
    term = preprocess(word)[0]
    if term in w2v.wv.key_to_index:
        print(f"{word:7} ->", [w for w, _ in w2v.wv.most_similar(term, topn=6)])

### Top-20 documents for each of our 5 queries (AND search + word2vec + cosine)

In [ ]:
w2v_results = {}
for q in test_queries.values():
    w2v_results[q] = rank(q, "w2v", top_k=20)
    print(f"\nQuery: {q}")
    print(w2v_results[q][["rank", "pid", "score", "title"]].to_string(index=False))

In [ ]:
pd.DataFrame([{"query": q,
               "w2v vs TF-IDF (overlap@20)": overlap_at_k(w2v_results[q]["pid"].tolist(),
                                                         rank(q, "tfidf", 20)["pid"].tolist(), k=20),
               "w2v vs BM25 (overlap@20)": overlap_at_k(w2v_results[q]["pid"].tolist(),
                                                       rank(q, "bm25", 20)["pid"].tolist(), k=20)}
              for q in test_queries.values()]).set_index("query")

**Comments.** With AND search every result already contains all the query words, so word2vec does not change *which* documents are returned, only their order. Averaging gives a high score to documents whose words are all "about the same thing" as the query (short, focused products); documents with a long description full of other words (fabric care, marketing text) get a diluted vector and move down. Word2vec does not look at how *rare* a word is: *men* weighs as much as *kurta* in the average, which is a disadvantage compared to TF-IDF/BM25.

---
# Evaluation of the four methods

We apply the Part 2 metrics to every method. As in Part 2, only labelled documents are kept, in ranking order, and relevant labelled documents not returned by the AND search count as missed.
- **Validation queries** (`validation_labels.csv`): 20 labelled documents per query.
- **Our 5 queries** (`my_labels.csv`): we labelled the top 20 of **TF-IDF** in Part 2. Other methods may bring documents we never judged — those are skipped — so this comparison slightly favours TF-IDF and must be read with care.

In [ ]:
def precision_at_k(rels, k):
    return sum(rels[:k]) / k


def avg_precision_at_k(rels, k, n_rel):
    hits, total = 0, 0.0
    for i, r in enumerate(rels[:k], start=1):
        if r:
            hits += 1
            total += hits / i
    return total / min(n_rel, k) if n_rel else 0.0


def reciprocal_rank(rels, k):
    for i, r in enumerate(rels[:k], start=1):
        if r:
            return 1 / i
    return 0.0


def ndcg_at_k(rels, k, n_rel):
    dcg = sum(r / math.log2(i + 1) for i, r in enumerate(rels[:k], start=1))
    ideal = sum(1 / math.log2(i + 1) for i in range(1, min(n_rel, k) + 1))
    return dcg / ideal if ideal else 0.0


ALL_METHODS = ["tfidf", "bm25", "ours", "w2v"]
K = 10


def evaluate(queries, judgments, k=K):
    # queries: {qid: text}; judgments: DataFrame with columns query_id, pid, label
    rows = []
    for qid, q in queries.items():
        judged = judgments[judgments["query_id"] == qid].set_index("pid")["label"]
        n_rel = int(judged.sum())
        for m in ALL_METHODS:
            rels = [int(judged[p]) for p in rank(q, m, top_k=None)["pid"] if p in judged.index]
            rows.append({"query": q, "method": m, f"P@{k}": precision_at_k(rels, k),
                         f"AP@{k}": avg_precision_at_k(rels, k, n_rel), f"RR@{k}": reciprocal_rank(rels, k),
                         f"NDCG@{k}": ndcg_at_k(rels, k, n_rel)})
    per_query = pd.DataFrame(rows).set_index(["query", "method"]).round(3)
    summary = (pd.DataFrame(rows).groupby("method")[[f"AP@{k}", f"RR@{k}", f"NDCG@{k}"]].mean()
               .rename(columns={f"AP@{k}": f"MAP@{k}", f"RR@{k}": f"MRR@{k}", f"NDCG@{k}": f"mean NDCG@{k}"})
               .loc[ALL_METHODS].round(3))
    return per_query, summary


validation_labels = pd.read_csv("../../data/validation_labels.csv").rename(columns={"labels": "label"})
validation_queries = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
val_table, val_summary = evaluate(validation_queries, validation_labels)
val_table

In [ ]:
val_summary

In [ ]:
my_labels = pd.read_csv("../part_2/my_labels.csv")
my_table, my_summary = evaluate(test_queries, my_labels)
my_table

In [ ]:
my_summary

---
# 3. Can we find a better representation than word2vec?

Averaging word2vec vectors has clear limits:
- **Word order is lost** (*shirt for men* = *men for shirt*).
- **Every word weighs the same**: very frequent words (*cotton*, *regular*, *men*) pull all vectors in the same direction, so many products end up very similar to each other.
- **Long texts are diluted**: a long description with many unrelated words hides the few words that matter.
- **One vector per word**: a word has the same vector in every context.

| Representation | How it works | Pros | Cons |
|---|---|---|---|
| **Weighted average** (TF-IDF or SIF weights) | Average of the word vectors weighted by IDF (or $a/(a+p(w))$), removing the common component | Cheap, reuses the same word2vec model, fixes "all words weigh the same" | Still ignores word order and context |
| **Doc2Vec** (PV-DM / PV-DBOW) | Learns one vector per document together with the word vectors | The document vector is trained directly, instead of averaged; same library (gensim) | New texts (the queries) need `infer_vector`, which is slow and slightly random; needs many documents and tuning; short texts (titles, 3-word queries) are hard |
| **Sentence embeddings** (Sentence-BERT, Sentence2Vec) | A pre-trained transformer encodes the whole sentence, word order and context included | Understands context and synonyms (*hoodie* ≈ *hooded sweatshirt*, *trainers* ≈ *shoes*); pre-trained on huge corpora, no training on our data needed; best quality for semantic search | Much heavier (a GPU helps), slower to index; not trained on our product vocabulary (Indian garments, truncated brands); harder to explain why a document ranks high |

**Our answer.** Yes: **sentence embeddings (Sentence-BERT)** are the better representation for this search engine. Our queries are short and the main problems we found in Part 2 were synonyms and product types, which need context. We would combine them with BM25 (*hybrid search*): BM25 keeps exact matches important (brands, colours, gender) and the embeddings add semantic similarity. If computing resources are limited, the cheapest improvement is an **IDF-weighted average** of word2vec vectors.

Below, a small Doc2Vec experiment for comparison.

In [ ]:
tagged = [TaggedDocument(tokens, [pid]) for pid, tokens in zip(df["pid"], df["tokens"])]
d2v = Doc2Vec(tagged, vector_size=100, window=5, min_count=2, epochs=20, workers=1, seed=42)


def d2v_rank(query, top_k=20):
    terms = preprocess(query)
    q_vec = d2v.infer_vector(terms, epochs=50)
    scores = {p: cosine(q_vec, d2v.dv[p]) for p in conjunctive_search(query)}
    best = sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:top_k]
    return pd.DataFrame({"rank": range(1, len(best) + 1), "pid": [p for p, _ in best],
                         "score": [round(s, 4) for _, s in best], "title": [titles[p] for p, _ in best]})


for q in test_queries.values():
    print(f"\nQuery: {q}")
    print(d2v_rank(q, top_k=10).to_string(index=False))